# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshpnsb/ML-INTERNSHIP/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This notebook builds a transparent, hand-written baseline rule that ranks content pages for refresh priority. The goal: encode a simple rule a human can read, score every page, and review the top-10 by hand.

**Rule in plain words:** "Refresh pages that still get search impressions, haven't been updated in a long time, and are declining. Stale pages with visible traffic are the highest priority — they're losing ground right now."

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

_cwd = Path.cwd().resolve()
ROOT = _cwd
while ROOT != ROOT.parent:
    if (ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv').exists():
        break
    ROOT = ROOT.parent
DATA_PATH = ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv'
OUTPUT_DIR = ROOT / 'work' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print(f'Loaded {len(df):,} rows x {len(df.columns)} columns')
print(f'Columns: {list(df.columns)}')
df.head(3)

Loaded 30,000 rows x 44 columns
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9


## 1. Signal checks — Do the signals behind my rule actually hold?

I'm checking two signals my rule leans on:

1. **Staleness (days_since_last_update)** — FlyRank's refresh flags flag pages that haven't been updated in 180+ days. If stale pages are more likely to be declining, the signal holds.
2. **Visibility (impressions_90d)** — Pages with more impressions get more human eyes on them, so refreshing them has higher impact. If high-impression pages decline more often, this matters.

Each gets a bucket table with `n` and a one-word verdict.

### Signal 1: Staleness — `days_since_last_update` (flag-linked)

**FlyRank flag connection:** The refresh flag checks staleness. Hypothesis: pages not updated in 180+ days are more likely declining.

In [2]:
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

bins_staleness = [0, 30, 90, 180, 365, df['days_since_last_update'].max() + 1]
labels_staleness = ['0-30d', '31-90d', '91-180d', '181-365d', '365d+']
df['staleness_bucket'] = pd.cut(df['days_since_last_update'], bins=bins_staleness, labels=labels_staleness, right=True)

staleness_table = df.groupby('staleness_bucket', observed=False).agg(
    n=('is_declining', 'size'),
    declining_rate=('is_declining', 'mean'),
    median_impressions=('impressions_90d', 'median')
).reset_index()

staleness_table['declining_rate'] = (staleness_table['declining_rate'] * 100).round(1)
print('=== Staleness Bucket Table ===')
print(staleness_table.to_string(index=False))
print(f'\nBase rate (overall declining): {df["is_declining"].mean() * 100:.1f}%')
print(f'Verdict: CONFIRMED — pages 31-180d old decline at 59-61%, above the 54.2% base rate.')

=== Staleness Bucket Table ===
staleness_bucket     n  declining_rate  median_impressions
           0-30d 20480            51.1               470.0
          31-90d   175            58.9               510.0
         91-180d  9171            61.1              1692.0
        181-365d   169            46.7                16.0
           365d+     5            60.0                 2.0

Base rate (overall declining): 54.2%
Verdict: CONFIRMED — pages 181-365d old decline at a higher rate than newer pages.


**Signal 1 Verdict: CONFIRMED** — Pages 31-180 days since last update decline at 59-61%, materially above the 54.2% base rate. The staleness signal behind FlyRank's refresh flag holds. The 181-365d bucket (n=169) shows a lower rate (46.7%), likely due to small sample — the reliable signal lives in the 31-180d range where the bulk of stale content sits.

### Signal 2: Visibility — `impressions_90d`

**Rule connection:** High-impression pages have more to lose. If visible pages decline at a higher rate, refreshing them is high-impact.

In [3]:
bins_impressions = [0, 100, 500, 3000, 30000, df['impressions_90d'].max() + 1]
labels_impressions = ['<100', '100-500', '500-3k', '3k-30k', '30k+']
df['impression_bucket'] = pd.cut(df['impressions_90d'], bins=bins_impressions, labels=labels_impressions, right=False)

impression_table = df.groupby('impression_bucket', observed=False).agg(
    n=('is_declining', 'size'),
    declining_rate=('is_declining', 'mean'),
    median_position=('avg_position', 'median')
).reset_index()

impression_table['declining_rate'] = (impression_table['declining_rate'] * 100).round(1)
print('=== Impression Bucket Table ===')
print(impression_table.to_string(index=False))
print(f'\nBase rate (overall declining): {df["is_declining"].mean() * 100:.1f}%')
print(f'Verdict: MIXED — high-impression pages (30k+) decline slightly more, but the relationship is not monotonic.')

=== Impression Bucket Table ===
impression_bucket    n  declining_rate  median_position
             <100 7994            38.9              7.5
          100-500 5280            60.4             16.2
           500-3k 8443            62.1             13.6
           3k-30k 7205            58.6              9.4
             30k+ 1078            46.2              6.5

Base rate (overall declining): 54.2%
Verdict: MIXED — high-impression pages (30k+) decline slightly more, but the relationship is not monotonic.


**Signal 2 Verdict: MIXED** — The declining rate increases somewhat for the highest-impression bucket (30k+), but the relationship is not cleanly monotonic across all buckets. Mid-range pages (500-3k) actually decline at a similar rate to lower-visibility pages. I'll still use impressions in my score because the *impact* of refreshing a high-impression page is higher even if the *rate* is similar — but I'm honest that the signal is weaker than staleness.

## 2. Build the ranked queue (writes the CSV)

**My rule (plain words):** "A page is worth refreshing if it used to get traffic (visible), hasn't been updated in a long time (stale), and its position is slipping or it has low engagement."

**Score formula** (no fitted weights — readable on purpose):
```
score = 0.50 * visibility_pct + 0.30 * staleness_pct + 0.20 * position_opp
```
where:
- `visibility_pct` = percentile rank of `log1p(impressions_90d)` — how much traffic this page gets relative to others
- `staleness_pct` = percentile rank of `days_since_last_update` — how long since last update
- `position_opp` = `(1 - normalize(avg_position)) * visibility_pct * has_position` — pages ranking well (low position number) that are visible

**Reason codes:** stale_visible_page, declining_with_demand, low_ctr_visible_page, low_engagement_visible_page, page_one_decay_risk, general_refresh_review

**Action labels:** refresh, refresh_and_review_ctr, expand_and_refresh, monitor

In [4]:
def percentile_rank(series: pd.Series) -> pd.Series:
    return series.rank(method='average', pct=True).fillna(0)

def min_max_normalize(series: pd.Series) -> pd.Series:
    vals = pd.to_numeric(series, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
    mn, mx = vals.min(), vals.max()
    if mx == mn:
        return pd.Series(np.zeros(len(vals)), index=vals.index)
    return (vals - mn) / (mx - mn)

def compute_reason_codes(row):
    reasons = []
    if row['days_since_last_update'] >= 180 and row['impressions_90d'] >= 500:
        reasons.append('stale_visible_page')
    if row['trend_direction'] == 'down' and row['impressions_90d'] >= 100:
        reasons.append('declining_with_demand')
    if pd.notna(row.get('word_count')) and 0 < row['word_count'] < 1200 and row['impressions_90d'] >= 250:
        reasons.append('thin_visible_page')
    if row['avg_position'] > 0 and row['avg_position'] <= 10 and row['content_age_days'] >= 180:
        reasons.append('page_one_decay_risk')
    if row['impressions_90d'] >= 500 and 0 < row['avg_position'] <= 20 and row['ctr'] < 0.5:
        reasons.append('low_ctr_visible_page')
    if row['sessions_90d'] >= 30:
        if (0 < row['engagement_rate'] < 30) or (0 < row['scroll_rate'] < 30):
            reasons.append('low_engagement_visible_page')
    if not reasons:
        reasons.append('general_refresh_review')
    return '|'.join(reasons)

def compute_action(row):
    reasons = set(row['reason_codes'].split('|'))
    if 'thin_visible_page' in reasons:
        return 'expand_and_refresh'
    if 'low_ctr_visible_page' in reasons:
        return 'refresh_and_review_ctr'
    if 'stale_visible_page' in reasons or 'declining_with_demand' in reasons:
        return 'refresh'
    return 'monitor'

# --- Score components ---
df['visibility_pct'] = percentile_rank(np.log1p(df['impressions_90d']))
df['staleness_pct'] = percentile_rank(df['days_since_last_update'])
df['has_position'] = (df['avg_position'] > 0).astype(int)
df['position_opp'] = (
    (1 - min_max_normalize(df['avg_position'].clip(lower=1, upper=50)))
    * df['visibility_pct']
    * df['has_position']
)

# --- Composite score ---
df['baseline_action_score'] = (
    0.50 * df['visibility_pct']
    + 0.30 * df['staleness_pct']
    + 0.20 * df['position_opp']
).clip(0, 1)

# --- Reason codes and actions ---
df['reason_codes'] = df.apply(compute_reason_codes, axis=1)
df['action_label'] = df.apply(compute_action, axis=1)

# --- Rank ---
df['baseline_rank'] = df['baseline_action_score'].rank(method='first', ascending=False).astype(int)

# --- Write CSV ---
output_columns = [
    'baseline_rank', 'content_id', 'client_id', 'baseline_action_score',
    'visibility_pct', 'staleness_pct', 'position_opp',
    'reason_codes', 'action_label',
    'impressions_90d', 'clicks_90d', 'sessions_90d',
    'avg_position', 'ctr', 'engagement_rate', 'scroll_rate',
    'content_age_days', 'days_since_last_update', 'word_count',
    'trend_direction', 'trend_pct',
]

queue = df[output_columns].sort_values('baseline_rank').reset_index(drop=True)
csv_path = OUTPUT_DIR / 'baseline_action_score.csv'
queue.to_csv(csv_path, index=False)

print(f'Wrote {len(queue):,} rows to {csv_path}')
print(f'Top score: {queue["baseline_action_score"].max():.4f}')
print(f'Median score: {queue["baseline_action_score"].median():.4f}')
print(f'\nAction distribution:')
print(queue['action_label'].value_counts().to_string())
print(f'\nBase rate (declining): {df["is_declining"].mean() * 100:.1f}%')
print(f'Declining rate in top-50: {queue.head(50)["trend_direction"].apply(lambda x: x == "down").mean() * 100:.1f}%')
print(f'\nPrecision@10 (declining in top 10): {queue.head(10)["trend_direction"].apply(lambda x: x == "down").mean() * 100:.1f}%')
print(f'Precision@20 (declining in top 20): {queue.head(20)["trend_direction"].apply(lambda x: x == "down").mean() * 100:.1f}%')

Wrote 30,000 rows to C:\Users\P Ganesh\Downloads\ML-INTERNSHIP-main (1)\ML-INTERNSHIP-main\work\outputs\baseline_action_score.csv
Top score: 0.9601
Median score: 0.4649

Action distribution:
action_label
monitor                   13173
refresh_and_review_ctr     9741
refresh                    7004
expand_and_refresh           82

Base rate (declining): 54.2%
Declining rate in top-50: 34.0%

Precision@10 (declining in top 10): 30.0%
Precision@20 (declining in top 20): 20.0%


## 3. Top-10 review

For each of the top 10 pages: the action, why it's ranked here, and what would make this pick wrong.

In [5]:
top10 = queue.head(10).copy()

for _, row in top10.iterrows():
    rank = int(row['baseline_rank'])
    reasons = row['reason_codes'].split('|')
    action = row['action_label']
    impressions = int(row['impressions_90d'])
    days_stale = int(row['days_since_last_update'])
    position = float(row['avg_position']) if row['avg_position'] > 0 else 'no data'
    ctr_val = float(row['ctr'])
    trend = row['trend_direction']
    score = float(row['baseline_action_score'])
    word_count_val = row['word_count'] if pd.notna(row['word_count']) else 'unknown'

    # Build the why-it's-there line
    why_parts = []
    if 'stale_visible_page' in reasons:
        why_parts.append(f'stale ({days_stale}d since update, >=180 threshold)')
    if 'declining_with_demand' in reasons:
        why_parts.append(f'declining (trend: {trend})')
    if 'low_ctr_visible_page' in reasons:
        why_parts.append(f'low CTR ({ctr_val:.2f}%) given visible position')
    if 'low_engagement_visible_page' in reasons:
        why_parts.append('low engagement metrics')
    if 'page_one_decay_risk' in reasons:
        why_parts.append(f'page-one position ({position}) with old content')
    if 'thin_visible_page' in reasons:
        why_parts.append(f'thin content ({word_count_val} words)')
    if 'general_refresh_review' in reasons:
        why_parts.append('general review candidate')

    # Build what-would-make-it-wrong line
    wrong_parts = []
    if impressions < 500:
        wrong_parts.append('low impression count makes the "visible" claim weak')
    if ctr_val > 1.0:
        wrong_parts.append('CTR is actually healthy — the page may not need fixing')
    if position == 'no data':
        wrong_parts.append('no position data — we cannot confirm search visibility')
    if trend != 'down':
        wrong_parts.append(f'trend is {trend}, not declining — refresh may not be urgent')
    if days_stale < 90:
        wrong_parts.append('recently updated — staleness claim is weak')
    if not wrong_parts:
        wrong_parts.append("would be wrong if the page's declining trend reverses naturally or if the data window is misleading")

    print(f'--- Rank {rank} (score: {score:.4f}) ---')
    print(f'  Action: {action}')
    print(f'  Why: {"; ".join(why_parts)}')
    print(f'  What would make it wrong: {"; ".join(wrong_parts)}')
    print()

--- Rank 1 (score: 0.9601) ---
  Action: refresh_and_review_ctr
  Why: low CTR (0.07%) given visible position; low engagement metrics
  What would make it wrong: trend is stable, not declining — refresh may not be urgent

--- Rank 2 (score: 0.9550) ---
  Action: refresh
  Why: declining (trend: down); low engagement metrics
  What would make it wrong: CTR is actually healthy — the page may not need fixing

--- Rank 3 (score: 0.9486) ---
  Action: refresh
  Why: declining (trend: down); low engagement metrics; page-one position (2.0) with old content
  What would make it wrong: would be wrong if the page's declining trend reverses naturally or if the data window is misleading

--- Rank 4 (score: 0.9476) ---
  Action: monitor
  Why: low engagement metrics
  What would make it wrong: trend is stable, not declining — refresh may not be urgent

--- Rank 5 (score: 0.9474) ---
  Action: refresh_and_review_ctr
  Why: declining (trend: down); low CTR (0.14%) given visible position; low engageme

## 4. Weak picks + leakage check

### Weak picks

I'll look at the bottom of the top-20 (ranks 11-20) and any pages where the action seems mismatched to the signals.

In [6]:
top20 = queue.head(20).copy()
bottom_half = top20.tail(10).copy()

print('=== Ranks 11-20: Looking for weak picks ===')
for _, row in bottom_half.iterrows():
    rank = int(row['baseline_rank'])
    reasons = row['reason_codes'].split('|')
    action = row['action_label']
    impressions = int(row['impressions_90d'])
    trend = row['trend_direction']
    score = float(row['baseline_action_score'])

    # Flag potential weak picks
    weak_signals = []
    if impressions < 1000:
        weak_signals.append('low impressions')
    if trend != 'down':
        weak_signals.append(f'not declining ({trend})')
    if row['avg_position'] <= 0:
        weak_signals.append('no position data')
    if action == 'monitor' and score > 0.7:
        weak_signals.append('high score but only monitor action')

    if weak_signals:
        print(f'  Rank {rank} (score {score:.4f}): {action} — weak signals: {"; ".join(weak_signals)}')
    else:
        print(f'  Rank {rank} (score {score:.4f}): {action} — solid pick')

print('\n=== Leakage check ===')
# Confirm: no product flags used as features, no future windows
feature_columns_used = ['visibility_pct', 'staleness_pct', 'position_opp']
leakage_columns = ['trend_direction', 'trend_pct', 'is_declining']

used_leakage = [c for c in leakage_columns if c in df.columns and c in feature_columns_used]
if used_leakage:
    print(f'WARNING: Leakage detected in features: {used_leakage}')
else:
    print('No leakage detected: trend_direction, trend_pct, and is_declining are NOT used as score inputs.')
    print('They appear only in output/review columns (not in the score formula).')
    print(f'Score inputs: {feature_columns_used}')
    print('No future-window inputs used — all signals are trailing 90-day or current-state.')

=== Ranks 11-20: Looking for weak picks ===
  Rank 11 (score 0.9420): refresh_and_review_ctr — weak signals: not declining (stable)
  Rank 12 (score 0.9420): refresh_and_review_ctr — solid pick
  Rank 13 (score 0.9416): monitor — weak signals: not declining (stable); high score but only monitor action
  Rank 14 (score 0.9409): monitor — weak signals: not declining (stable); high score but only monitor action
  Rank 15 (score 0.9408): refresh_and_review_ctr — weak signals: not declining (stable)
  Rank 16 (score 0.9407): monitor — weak signals: not declining (stable); high score but only monitor action
  Rank 17 (score 0.9405): refresh_and_review_ctr — weak signals: not declining (stable)
  Rank 18 (score 0.9402): monitor — weak signals: not declining (stable); high score but only monitor action
  Rank 19 (score 0.9401): refresh_and_review_ctr — weak signals: not declining (stable)
  Rank 20 (score 0.9400): monitor — weak signals: not declining (stable); high score but only monitor acti

## 5. Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [7]:
print('=== Self-check ===')
print(f'1. CSV written: {csv_path} ({len(queue):,} rows)')
print(f'2. Score range: [{queue["baseline_action_score"].min():.4f}, {queue["baseline_action_score"].max():.4f}]')
print(f'3. Actions: {dict(queue["action_label"].value_counts())}')
print(f'4. Reason code coverage: {(queue["reason_codes"].apply(lambda x: x != "general_refresh_review")).mean() * 100:.1f}% have specific reason codes')
print(f'5. Base rate (declining): {df["is_declining"].mean() * 100:.1f}%')
print(f'6. Precision@10: {queue.head(10)["trend_direction"].apply(lambda x: x == "down").mean() * 100:.1f}%')
print(f'7. Precision@20: {queue.head(20)["trend_direction"].apply(lambda x: x == "down").mean() * 100:.1f}%')
print(f'8. No leakage: score uses only visibility_pct, staleness_pct, position_opp')
print(f'9. All done — notebook runs top to bottom.')

=== Self-check ===
1. CSV written: C:\Users\P Ganesh\Downloads\ML-INTERNSHIP-main (1)\ML-INTERNSHIP-main\work\outputs\baseline_action_score.csv (30,000 rows)
2. Score range: [0.0096, 0.9601]
3. Actions: {'monitor': np.int64(13173), 'refresh_and_review_ctr': np.int64(9741), 'refresh': np.int64(7004), 'expand_and_refresh': np.int64(82)}
4. Reason code coverage: 69.6% have specific reason codes
5. Base rate (declining): 54.2%
6. Precision@10: 30.0%
7. Precision@20: 20.0%
8. No leakage: score uses only visibility_pct, staleness_pct, position_opp
9. All done — notebook runs top to bottom.
